In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import multiprocessing
import gc
from joblib import Parallel, delayed
from scipy.stats import norm
from tqdm import tqdm

# ML e Métricas
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, recall_score, f1_score, roc_auc_score, 
                             precision_score, matthews_corrcoef, precision_recall_curve, 
                             auc, average_precision_score)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
from cleverhans.tf2.attacks.carlini_wagner_l2 import carlini_wagner_l2 # >>> NOVO: Import do C&W <<<

# Modelos Customizados
if 'libs' not in sys.path: sys.path.append('libs')
import importlib
import libs.bloom_filter
import libs.wisard
importlib.reload(libs.bloom_filter)
importlib.reload(libs.wisard)
from libs.wisard import WiSARD
from art.estimators.classification import KerasClassifier
from art.attacks.evasion import HopSkipJump
from tensorflow.keras.layers import Lambda

# ==========================================
# PAINEL DE CONTROLE DO EXPERIMENTO
# ==========================================
# DATASETS_TO_RUN = ['Bot-IoT', 'UNSW-NB15', 'CICIDS', 'Edge-IIoT', 'ToN-IoT']
DATASETS_TO_RUN = ['Bot-IoT', 'UNSW-NB15', 'CICIDS']
RUN_BINARY = True         
RUN_MULTICLASS = False      

ENCODING_TYPES = ['linear', 'gaussian', 'distributive']
# ENCODING_TYPES = ['linear']

# ATTACKS_TO_RUN = ['FGSM', 'RANDOM_LINF', 'RANDOM_L2', 'C&W', 'HSJA', 'BPDA']
ATTACKS_TO_RUN = ['HSJA', 'BPDA']
EPSILON_LINF = 0.3   
EPSILON_L2 = 3.0     

N_JOBS = -1
os.makedirs('relatorios final', exist_ok=True)
os.makedirs('curves_data', exist_ok=True) 

# # Pastas "Seguras" para não subscrever os seus dados originais!
# PASTA_RELATORIOS = 'relatorios_teste_rapido'
# PASTA_CURVAS = 'curves_teste_rapido'
# os.makedirs(PASTA_RELATORIOS, exist_ok=True)
# os.makedirs(PASTA_CURVAS, exist_ok=True)

print(f"Datasets Selecionados: {DATASETS_TO_RUN}")
print(f"Modos Ativados: Binário={RUN_BINARY} | Multiclasse={RUN_MULTICLASS}")
print(f"Ataques Ativados: {ATTACKS_TO_RUN}")
print(f"Binarizações: {ENCODING_TYPES}")

c:\Users\root.REDE-LAGESED\Desktop\Trabalho\.venv\Lib\site-packages\art\estimators\certification\__init__.py:30: UserWarning: PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality
  warnings.warn("PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality")


Datasets Selecionados: ['Bot-IoT', 'UNSW-NB15', 'CICIDS', 'Edge-IIoT', 'ToN-IoT']
Modos Ativados: Binário=True | Multiclasse=False
Ataques Ativados: ['HSJA', 'BPDA']
Binarizações: ['linear', 'gaussian', 'distributive']


In [2]:
# ==========================================
# FUNÇÕES DE MODELAÇÃO E MÉTRICAS
# ==========================================
def build_and_train_mlp(X, y_cat, num_classes):
    inputs = Input(shape=(X.shape[1],))
    x = Dense(256, activation='relu')(inputs)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.4)(x)
    logits = Dense(num_classes, name='logits')(x)
    outputs = Activation('softmax')(logits)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])
    model.fit(X, y_cat, batch_size=64, epochs=5, verbose=0)
    return model

def process_data_vectorized_sequential(data, resolution, enc_type, custom_thresholds=None, chunk_size=20000):
    n_samples, n_features = data.shape
    result = np.empty((n_samples, n_features * resolution), dtype=np.int8)
    
    if enc_type == 'linear':
        indices = np.arange(resolution, dtype=np.int8)
        
    total_chunks = (n_samples + chunk_size - 1) // chunk_size
    
    for i in range(total_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, n_samples)
        chunk = data[start:end]
        
        if enc_type in ['gaussian', 'distributive']:
            bits = (chunk[:, :, None] >= custom_thresholds[None, :, :]).astype(np.int8)
        elif enc_type == 'linear':
            chunk_clipped = np.clip(chunk, 0.0, 1.0)
            limits = (chunk_clipped * resolution).astype(np.int8)
            bits = (limits[:, :, None] > indices[None, None, :]).astype(np.int8)
            
        result[start:end] = bits.reshape(chunk.shape[0], -1)
    return result

def predict_worker_dynamic_with_prob(model, chunk):
    preds = []
    probs = []
    pad_zeros = model.pad_zeros
    input_order = model.input_order
    discriminators = model.discriminators
    num_classes = len(discriminators)
    
    for sample in chunk:
        xv_padded = np.pad(np.array(sample), (0, pad_zeros))[input_order]
        b = 1
        last_tie = [0]
        first_responses = None
        
        while b <= 200:
            model.set_bleaching(b)
            responses = np.array([d.predict(xv_padded) for d in discriminators], dtype=float)
            
            if b == 1:
                first_responses = responses.copy()
                
            max_res = responses.max()
            if max_res == 0: break 
            winners = np.where(responses == max_res)[0]
            if len(winners) == 1:
                last_tie = winners
                break
            else:
                last_tie = winners
                b += max(1, int(b * 0.2))
                
        pred_class = int(last_tie[0])
        preds.append(pred_class)
        
        if first_responses is not None and first_responses.sum() > 0:
            prob = first_responses / first_responses.sum()
        else:
            prob = np.zeros(num_classes)
            prob[pred_class] = 1.0
        probs.append(prob)
        
    return preds, probs

def true_parallel_evaluate_worker_bloom(num_inputs, num_classes, addr, bloom, h, X_train_np, y_train_curr, X_clean_chunk_np, adv_chunks_dict_np):
    # 1. Instanciar e Treinar a rede (Isolado no worker - Usa 1 núcleo a 100%)
    t0 = time.time()
    model = WiSARD(num_inputs, num_classes, addr, bloom, h)
    for i in range(len(X_train_np)):
        model.train(X_train_np[i], int(y_train_curr[i]))
    train_time = time.time() - t0
    
    # 2. Classificar a fatia de dados limpos (com o desempate dinâmico)
    t1 = time.time()
    yp_clean, yp_clean_prob = predict_worker_dynamic_with_prob(model, X_clean_chunk_np)
    infer_time_clean = time.time() - t1
    
    # 3. Classificar as fatias dos ataques adversariais
    res_adv = {}
    prob_adv = {}
    infer_time_adv = {}
    for atk_name, chunk_np in adv_chunks_dict_np.items():
        t2 = time.time()
        yp_a, yp_a_prob = predict_worker_dynamic_with_prob(model, chunk_np)
        res_adv[atk_name] = yp_a
        prob_adv[atk_name] = yp_a_prob
        infer_time_adv[atk_name] = time.time() - t2
        
    return yp_clean, yp_clean_prob, res_adv, prob_adv, train_time, infer_time_adv

def calculate_miss_rates(y_true, y_pred, y_prob, context_name, class_names):
    metrics = {}
    try:
        normal_idx = next(i for i, name in enumerate(class_names) if 'normal' in str(name).lower())
    except StopIteration:
        normal_idx = 0
        
    is_binary = len(class_names) == 2
    avg_type = 'binary' if is_binary else 'weighted'
    pos_label = 1 if is_binary else None
    
    metrics[f'{context_name}_Acc'] = accuracy_score(y_true, y_pred)
    metrics[f'{context_name}_Precision'] = precision_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_Recall'] = recall_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_F1'] = f1_score(y_true, y_pred, average=avg_type, pos_label=pos_label, zero_division=0)
    metrics[f'{context_name}_MCC'] = matthews_corrcoef(y_true, y_pred)
    
    mask_normal = (y_true == normal_idx)
    mask_attack = (y_true != normal_idx)
    
    if np.sum(mask_normal) > 0:
        metrics[f'{context_name}_FAR'] = np.sum((y_pred != normal_idx) & mask_normal) / np.sum(mask_normal)
    else:
        metrics[f'{context_name}_FAR'] = 0.0

    if np.sum(mask_attack) > 0:
        metrics[f'{context_name}_ASR'] = np.sum((y_pred == normal_idx) & mask_attack) / np.sum(mask_attack)
    else:
        metrics[f'{context_name}_ASR'] = 0.0

    try:
        if is_binary:
            prob_positive = y_prob[:, 1] if len(y_prob.shape) > 1 else y_prob
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, prob_positive)
            prec, rec, _ = precision_recall_curve(y_true, prob_positive)
            metrics[f'{context_name}_PR_AUC'] = auc(rec, prec)
        else:
            metrics[f'{context_name}_AUC'] = roc_auc_score(y_true, y_prob, multi_class='ovr')
            y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
            metrics[f'{context_name}_PR_AUC'] = average_precision_score(y_true_bin, y_prob, average="macro")
    except Exception as e:
        metrics[f'{context_name}_AUC'] = 0.0
        metrics[f'{context_name}_PR_AUC'] = 0.0
    
    if not is_binary:
        metrics[f'{context_name}_F1_Macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    for idx, name in enumerate(class_names):
        if idx == normal_idx: continue
        mask_t = (y_true == idx)
        if np.sum(mask_t) > 0:
            metrics[f'{context_name}_Miss_{name}'] = np.sum((y_pred == normal_idx) & mask_t) / np.sum(mask_t)
        else:
            metrics[f'{context_name}_Miss_{name}'] = 0.0
            
    return metrics

In [3]:
# ==========================================
# LOOP PRINCIPAL DO EXPERIMENTO
# ==========================================
for dataset_name in DATASETS_TO_RUN:
    print(f"\n{'='*50}")
    print(f">>> A INICIAR EXPERIMENTOS: {dataset_name.upper()}")
    print(f"{'='*50}")
            
    # 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
    if dataset_name == 'Bot-IoT':
        df_train = pd.read_csv("data2/BotIoT_training-set.csv")
        df_test = pd.read_csv("data2/BotIoT_testing-set.csv")
    elif dataset_name == 'UNSW-NB15':
        df_train = pd.read_csv("data/UNSW_NB15_training-set.csv")
        df_test = pd.read_csv("data/UNSW_NB15_testing-set.csv")
    elif dataset_name == 'CICIDS':
        df_train = pd.read_csv("data3/CICIDS_training-set.csv")
        df_test = pd.read_csv("data3/CICIDS_testing-set.csv")
    elif dataset_name == 'ToN-IoT':
            df_train = pd.read_csv("data4/ToN-IoT_training-set.csv")
            df_test = pd.read_csv("data4/ToN-IoT_testing-set.csv")
    elif dataset_name == 'Edge-IIoT':
        df_train = pd.read_csv("data5/Edge-IIoT_training-set.csv")
        df_test = pd.read_csv("data5/Edge-IIoT_testing-set.csv")

    # # >>> ADICIONE ESTAS DUAS LINHAS AQUI <<<
    # # Reduz drasticamente para vermos o pipeline funcionar em segundos
    # df_train = df_train.sample(100, random_state=42)
    # df_test = df_test.sample(20, random_state=42)

    for df in [df_train, df_test]:
        if 'id' in df.columns: df.drop(columns=['id'], inplace=True)

    y_train_bin = df_train['label'].values
    y_test_bin = df_test['label'].values
    class_names_bin = ['Normal', 'Attack']

    df_train['attack_cat'] = df_train['attack_cat'].astype(str).str.strip().str.lower()
    df_test['attack_cat'] = df_test['attack_cat'].astype(str).str.strip().str.lower()
    le = LabelEncoder()
    le.fit(pd.concat([df_train['attack_cat'], df_test['attack_cat']]))
    y_train_multi = le.transform(df_train['attack_cat'])
    y_test_multi = le.transform(df_test['attack_cat'])
    class_names_multi = le.classes_

    df_train.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')
    df_test.drop(columns=['label', 'attack_cat'], inplace=True, errors='ignore')

    # 1. Encontrar colunas categóricas e numéricas
    categorical_cols = df_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    numerical_cols = df_train.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

    # 2. Blindagem Total: Preencher Nulos Ocultos antes da conversão
    for col in categorical_cols:
        df_train[col] = df_train[col].fillna('unknown').astype(str)
        df_test[col] = df_test[col].fillna('unknown').astype(str)

    for col in numerical_cols:
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce').fillna(0.0)
        df_test[col] = pd.to_numeric(df_test[col], errors='coerce').fillna(0.0)

    # 3. Criar o Preprocessor
    preprocessor = ColumnTransformer([
        ('num', MinMaxScaler(feature_range=(0,1)), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

    print(">>> A aplicar Scaling e One-Hot Encoding...")
    preprocessor.fit(df_train)
    X_train = preprocessor.transform(df_train).astype('float32')
    X_test = preprocessor.transform(df_test).astype('float32')

    del df_train, df_test
    gc.collect()

    # 2. GERAÇÃO DOS ATAQUES ADVERSARIAIS E RUÍDOS
    attacks_dict_bin = {}
    attacks_dict_multi = {}

    if 'FGSM' in ATTACKS_TO_RUN:
        print(">>> A gerar Ataque FGSM (Caixa-Branca Transferida)...")
        if RUN_BINARY:
            mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            logits_bin = Model(inputs=mlp_bin.input, outputs=mlp_bin.get_layer('logits').output)
            attacks_dict_bin['FGSM'] = fast_gradient_method(logits_bin, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_bin, logits_bin 
        
        if RUN_MULTICLASS:
            mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
            logits_multi = Model(inputs=mlp_multi.input, outputs=mlp_multi.get_layer('logits').output)
            attacks_dict_multi['FGSM'] = fast_gradient_method(logits_multi, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_multi, logits_multi 
        gc.collect()

    if 'RANDOM_LINF' in ATTACKS_TO_RUN:
        print(f">>> A gerar Ruído Aleatório L_infinito (Eps={EPSILON_LINF})...")
        noise = np.random.uniform(-EPSILON_LINF, EPSILON_LINF, X_test.shape).astype('float32')
        if RUN_BINARY: attacks_dict_bin['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)
        if RUN_MULTICLASS: attacks_dict_multi['RANDOM_LINF'] = np.clip(X_test + noise, 0.0, 1.0)
        del noise

    if 'RANDOM_L2' in ATTACKS_TO_RUN:
        print(f">>> A gerar Ruído Aleatório L_2 (Eps={EPSILON_L2})...")
        noise = np.random.normal(0, 1, X_test.shape).astype('float32')
        norms = np.linalg.norm(noise, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10
        noise = noise * (EPSILON_L2 / norms)
        if RUN_BINARY: attacks_dict_bin['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)
        if RUN_MULTICLASS: attacks_dict_multi['RANDOM_L2'] = np.clip(X_test + noise, 0.0, 1.0)
        del noise

    # >>> NOVO: BLOCO DO ATAQUE C&W <<<
    if 'C&W' in ATTACKS_TO_RUN:
        print(">>> A gerar Ataque C&W L2 (Aviso: Otimização pesada, processando em lotes)...")
        
        # Função interna para processar o C&W em lotes com Barra de Progresso
        # PRESAL-WS20 OTIMIZAÇÃO: Aumento do batch_size para 2500 graças aos 256GB de RAM
        def generate_cw_in_batches(logits_model, X_np, n_classes, batch_size=2500):
            adv_x = []
            total_batches = (len(X_np) + batch_size - 1) // batch_size
            
            for i in tqdm(range(total_batches), desc="Gerando Lotes C&W", unit="lote"):
                start_idx = i * batch_size
                end_idx = min((i + 1) * batch_size, len(X_np))
                
                x_batch = tf.convert_to_tensor(X_np[start_idx:end_idx], dtype=tf.float32)
                x_batch = tf.clip_by_value(x_batch, clip_value_min=0.0, clip_value_max=1.0)
                
                preds = logits_model(x_batch)
                y_one_hot = tf.one_hot(tf.argmax(preds, axis=1), depth=n_classes)
                
                adv_batch = carlini_wagner_l2(
                    logits_model, 
                    x_batch, 
                    y=y_one_hot,
                    batch_size=x_batch.shape[0], 
                    clip_min=0.0, 
                    clip_max=1.0,
                    max_iterations=100 
                )
                
                adv_x.append(adv_batch)
                
            return np.vstack(adv_x)

        if RUN_BINARY:
            print("   -> Treinando modelo base para o C&W (Binário)...")
            mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            logits_bin = Model(inputs=mlp_bin.input, outputs=mlp_bin.get_layer('logits').output)
            attacks_dict_bin['C&W'] = generate_cw_in_batches(logits_bin, X_test, n_classes=2)
            del mlp_bin, logits_bin
        
        if RUN_MULTICLASS:
            print("   -> Treinando modelo base para o C&W (Multiclasse)...")
            mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
            logits_multi = Model(inputs=mlp_multi.input, outputs=mlp_multi.get_layer('logits').output)
            attacks_dict_multi['C&W'] = generate_cw_in_batches(logits_multi, X_test, n_classes=len(class_names_multi))
            del mlp_multi, logits_multi

    # =================================================================
    # >>> NOVOS ATAQUES: HSJA (ART) e BPDA (Custom Gradient)
    # =================================================================

    if 'HSJA' in ATTACKS_TO_RUN:
        print(">>> A gerar HopSkipJumpAttack (Decision-Based)...")
        if RUN_BINARY:
            mlp_bin = build_and_train_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            art_classifier_bin = KerasClassifier(model=mlp_bin, clip_values=(0.0, 1.0), use_logits=False)
            # max_iter reduzido para poupar tempo. Aumente para 50 se quiser ataques mais perfeitos
            hsja = HopSkipJump(classifier=art_classifier_bin, norm=np.inf, max_iter=50, max_eval=1000, init_eval=100)
            attacks_dict_bin['HSJA'] = hsja.generate(x=X_test)
            del mlp_bin, art_classifier_bin
            
        if RUN_MULTICLASS:
            mlp_multi = build_and_train_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
            art_classifier_multi = KerasClassifier(model=mlp_multi, clip_values=(0.0, 1.0), use_logits=False)
            hsja_multi = HopSkipJump(classifier=art_classifier_multi, norm=np.inf, max_iter=50, max_eval=1000, init_eval=100)
            attacks_dict_multi['HSJA'] = hsja_multi.generate(x=X_test)
            del mlp_multi, art_classifier_multi

    if 'BPDA' in ATTACKS_TO_RUN:
        print(">>> A gerar BPDA (Aproximação Diferenciável do Backward Pass)...")
        
        # 1. Definir uma Binarização com STE (Straight-Through Estimator)
        @tf.custom_gradient
        def binarize_ste(x):
            forward = tf.cast(x >= 0.5, tf.float32)
            def backward(dy):
                return dy 
            return forward, backward

        # 2. Criar um MLP que inclui o simulador de binarização da WiSARD
        def build_bpda_mlp(X, y_cat, num_classes):
            inputs = Input(shape=(X.shape[1],))
            # >>> NOVO: Envolver a função numa camada Lambda <<<
            bin_inputs = Lambda(binarize_ste)(inputs) 
            x = Dense(256, activation='relu')(bin_inputs)
            x = Dropout(0.4)(x)
            x = Dense(128, activation='relu')(x)
            x = Dropout(0.4)(x)
            logits = Dense(num_classes, name='logits')(x)
            outputs = Activation('softmax')(logits)
            model = Model(inputs=inputs, outputs=outputs)
            model.compile(loss='categorical_crossentropy', optimizer=Adam(0.001), metrics=['accuracy'])
            model.fit(X, y_cat, batch_size=64, epochs=5, verbose=0)
            return model

        if RUN_BINARY:
            mlp_bpda_bin = build_bpda_mlp(X_train, to_categorical(y_train_bin, 2), 2)
            logits_bpda_bin = Model(inputs=mlp_bpda_bin.input, outputs=mlp_bpda_bin.get_layer('logits').output)
            attacks_dict_bin['BPDA'] = fast_gradient_method(logits_bpda_bin, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_bpda_bin, logits_bpda_bin

        if RUN_MULTICLASS:
            mlp_bpda_multi = build_bpda_mlp(X_train, to_categorical(y_train_multi, len(class_names_multi)), len(class_names_multi))
            logits_bpda_multi = Model(inputs=mlp_bpda_multi.input, outputs=mlp_bpda_multi.get_layer('logits').output)
            attacks_dict_multi['BPDA'] = fast_gradient_method(logits_bpda_multi, tf.convert_to_tensor(X_test), EPSILON_LINF, np.inf, clip_min=0.0, clip_max=1.0).numpy()
            del mlp_bpda_multi, logits_bpda_multi
            
    gc.collect()

    # 3. HIPERPARÂMETROS
    param_grid = {
        'resolution': [1, 2, 4, 8, 10],
        'addressSize': [5, 10, 15, 20],
        'bloomSize': [1024, 2048],
        'hashes': [2, 3]
    }

    # # 3. HIPERPARÂMETROS (Reduzidos para Teste)
    # param_grid = {
    #     'resolution': [4],
    #     'addressSize': [10],
    #     'bloomSize': [1024], # Se for a Standard WiSARD, apague isto
    #     'hashes': [2]        # Se for a Standard WiSARD, apague isto
    # }

    # Pré-cálculo para termômetro gaussiano
    X_mean = X_train.mean(axis=0)
    X_std = X_train.std(axis=0)
    X_std[X_std == 0] = 1e-8 

    # 4. LOOP DA BLOOM WISARD
    for enc_type in ENCODING_TYPES:
        for res in param_grid['resolution']:
            print(f"\n[{dataset_name} | {enc_type.upper()} | Res={res}] Binarização C...")
            
            custom_thresh = None
            
            if enc_type == 'gaussian':
                skews = [norm.ppf((i+1)/(res+1)) for i in range(res)]
                custom_thresh = X_mean[:, None] + (X_std[:, None] * skews)
            
            elif enc_type == 'distributive':
                percentiles = np.linspace(0, 100, res + 2)[1:-1]
                custom_thresh = np.percentile(X_train, percentiles, axis=0).T 

            # Binarizando os dados limpos
            X_train_bin = process_data_vectorized_sequential(X_train, res, enc_type, custom_thresh)
            X_test_bin = process_data_vectorized_sequential(X_test, res, enc_type, custom_thresh)
            num_inputs = len(X_train_bin[0])
            
            # Binarizando matrizes de ataque ativas
            bin_adv_dict = {}
            if RUN_BINARY:
                for atk_name, X_adv_matrix in attacks_dict_bin.items():
                    bin_adv_dict[atk_name] = process_data_vectorized_sequential(X_adv_matrix, res, enc_type, custom_thresh)
                    
            multi_adv_dict = {}
            if RUN_MULTICLASS:
                for atk_name, X_adv_matrix in attacks_dict_multi.items():
                    multi_adv_dict[atk_name] = process_data_vectorized_sequential(X_adv_matrix, res, enc_type, custom_thresh)

            modes_to_run = []
            if RUN_BINARY: modes_to_run.append('binary')
            if RUN_MULTICLASS: modes_to_run.append('multiclass')

            for mode in modes_to_run:
                if mode == 'binary':
                    y_train_curr, y_test_curr, class_names_curr = y_train_bin, y_test_bin, class_names_bin
                    active_adv_dict = bin_adv_dict
                else:
                    y_train_curr, y_test_curr, class_names_curr = y_train_multi, y_test_multi, class_names_multi
                    active_adv_dict = multi_adv_dict

                num_classes = len(class_names_curr)
                csv_name = f'relatorios final/bloom_wisard_{dataset_name}_{mode}_{enc_type}.csv'

                # Altere para:
                # csv_name = f'{PASTA_RELATORIOS}/bloom_wisard_{dataset_name}_{mode}_{enc_type}.csv'
                
                for addr in param_grid['addressSize']:
                    for bloom in param_grid['bloomSize']:
                        for h in param_grid['hashes']:
                            print(f"     [Addr={addr} | Bloom={bloom} | Mode={mode} | Hash={h}] A treinar e avaliar WiSARD (True Multiprocessing)...")
                            
                            n_jobs = max(1, multiprocessing.cpu_count() - 1)
                            
                            # 1. Cortar os dados em fatias exatas
                            clean_chunks = np.array_split(X_test_bin, n_jobs)
                            
                            adv_chunks_list = [] 
                            for i in range(n_jobs):
                                chunk_dict = {}
                                for atk_name, X_adv_matrix in active_adv_dict.items():
                                    chunk_dict[atk_name] = np.array_split(X_adv_matrix, n_jobs)[i]
                                adv_chunks_list.append(chunk_dict)
                                
                            # 2. Iniciar Paralelismo Absoluto
                            # Os arrays Numpy usam Memmap (zero cópias pesadas na RAM), 
                            # e cada núcleo treina a sua Bloom WiSARD sem transferências lentas de Pickle.
                            results = Parallel(n_jobs=n_jobs, backend='loky')(
                                delayed(true_parallel_evaluate_worker_bloom)(
                                    num_inputs, num_classes, addr, bloom, h,
                                    X_train_bin, y_train_curr, clean_chunks[i], adv_chunks_list[i]
                                ) for i in range(n_jobs)
                            )
                            
                            # 3. Juntar Resultados
                            yp_clean = []
                            yp_clean_prob = []
                            yp_adv_dict = {atk: [] for atk in active_adv_dict.keys()}
                            yp_adv_prob_dict = {atk: [] for atk in active_adv_dict.keys()}
                            
                            for res_c, prob_c, res_a, prob_a, t_train, t_adv_dict in results:
                                yp_clean.extend(res_c)
                                yp_clean_prob.extend(prob_c)
                                for atk in active_adv_dict.keys():
                                    yp_adv_dict[atk].extend(res_a[atk])
                                    yp_adv_prob_dict[atk].extend(prob_a[atk])
                                    
                            yp_clean = np.array(yp_clean)
                            yp_clean_prob = np.array(yp_clean_prob)
                            
                            m_clean = calculate_miss_rates(y_test_curr, yp_clean, yp_clean_prob, "Clean", class_names_curr)
                            
                            # Como treinamos em paralelo, pegamos o tempo de treino do worker mais lento
                            train_time_real = max([r[4] for r in results])
                            
                            for atk_name in active_adv_dict.keys():
                                yp_adv = np.array(yp_adv_dict[atk_name])
                                yp_adv_prob = np.array(yp_adv_prob_dict[atk_name])
                                infer_time_adv_real = max([r[5][atk_name] for r in results])
                                
                                m_adv = calculate_miss_rates(y_test_curr, yp_adv, yp_adv_prob, "Adv", class_names_curr)
                                acc_drop = m_clean['Clean_Acc'] - m_adv['Adv_Acc']
                                
                                row = {
                                    'Dataset': dataset_name,
                                    'Attack': atk_name,
                                    'Resolution': res, 
                                    'AddressSize': addr, 
                                    'BloomSize': bloom,
                                    'Hash': h, 
                                    'TrainTime_s': train_time_real, 
                                    'InferTime_Adv_s': infer_time_adv_real,    
                                    'Acc_Drop_pp': acc_drop*100
                                }
                                row.update(m_clean)
                                row.update(m_adv)
                                
                                row_df = pd.DataFrame([row])
                                file_exists = os.path.exists(csv_name)
                                row_df.to_csv(csv_name, mode='a', header=not file_exists, index=False)
                                
                                file_tag = f"bloom_wisard_{dataset_name}_{mode}_{enc_type}_{atk_name}_R{res}_A{addr}_B{bloom}_H{h}"

                                # np.savez(f"{PASTA_CURVAS}/{file_tag}.npz", 
                                np.savez(f"curves_data/{file_tag}.npz",
                                        model_name=f"Bloom WiSARD ({enc_type})",
                                        attack_name=atk_name,
                                        y_true=y_test_curr, 
                                        y_prob_clean=yp_clean_prob, 
                                        y_prob_adv=yp_adv_prob,
                                        class_names=class_names_curr)
                                
                            del results, clean_chunks, adv_chunks_list, yp_clean, yp_clean_prob, yp_adv_dict, yp_adv_prob_dict
                            gc.collect()

            # <-- ALINHADO COM O FOR DA RESOLUÇÃO
            del X_train_bin, X_test_bin, bin_adv_dict, multi_adv_dict
            gc.collect()

    # <-- ALINHADO COM O FOR DO DATASET
    del attacks_dict_bin, attacks_dict_multi, X_train, X_test
    gc.collect()

# <-- ALINHADO À MARGEM ESQUERDA
print("\n🚀 EXPERIMENTO CONCLUÍDO COM SUCESSO!")


>>> A INICIAR EXPERIMENTOS: BOT-IOT
>>> A aplicar Scaling e One-Hot Encoding...
>>> A gerar HopSkipJumpAttack (Decision-Based)...


HopSkipJump:   0%|          | 0/110056 [00:00<?, ?it/s]

>>> A gerar BPDA (Aproximação Diferenciável do Backward Pass)...


[Bot-IoT | LINEAR | Res=1] Binarização C...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=15 | Bloom=1024 | Mode=binary | Hash=2] A treinar 

HopSkipJump:   0%|          | 0/82332 [00:00<?, ?it/s]

>>> A gerar BPDA (Aproximação Diferenciável do Backward Pass)...

[UNSW-NB15 | LINEAR | Res=1] Binarização C...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=15 | Bloom=1024 | Mode=binary | Hash=2] A treinar

...

>>> A gerar BPDA (Aproximação Diferenciável do Backward Pass)...

[CICIDS | LINEAR | Res=1] Binarização C...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=15 | Bloom=1024 | Mode=binary | Hash=2] A treinar e 

HopSkipJump:   0%|          | 0/443841 [00:00<?, ?it/s]

>>> A gerar BPDA (Aproximação Diferenciável do Backward Pass)...

[Edge-IIoT | LINEAR | Res=1] Binarização C...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=15 | Bloom=1024 | Mode=binary | Hash=2] A treinar

HopSkipJump:   0%|          | 0/42209 [00:00<?, ?it/s]

>>> A gerar BPDA (Aproximação Diferenciável do Backward Pass)...

[ToN-IoT | LINEAR | Res=1] Binarização C...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=5 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=1024 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=2] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=10 | Bloom=2048 | Mode=binary | Hash=3] A treinar e avaliar WiSARD (True Multiprocessing)...
     [Addr=15 | Bloom=1024 | Mode=binary | Hash=2] A treinar e